# 7 · Inspecting & Cleaning Data
*Intro to Python for Scientists & Public Health Professionals*

Real data arrives messy: wrong types, duplicate rows, missing values, sentinel codes like `?`, inconsistent categories, and outliers. This lesson is the workflow for turning a raw load into something analysis-ready, using the same diabetes dataset we first met in the pandas intro.

A note on plots: we use a handful of quick charts here purely to *guide cleaning decisions*. Producing polished, publication-quality visuals is the focus of the separate Data Visualization course.

### By the end of this notebook you can
- Inspect a dataset (`head`, `info`, `describe`)
- Remove duplicates and fix data types
- Find and handle missing data, including sentinel values
- Standardize messy categorical values
- Spot and remove outliers, and drop columns/rows you don't need

### Agenda
1. First look
2. Duplicates
3. Data types
4. Renaming columns
5. Missing data
6. Imputing
7. Fixing categorical values
8. Distributions & outliers
9. Dropping columns & rows

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_URL = "https://raw.githubusercontent.com/jimcody2014/2026-python-data/refs/heads/main"
df = pd.read_csv(f"{BASE_URL}/diabetes_inspect.csv")
df.shape

## 1. First look

Three quick views before touching anything: the first rows, the column/dtype summary, and numeric summary stats.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()    # summary stats for the numeric columns

## 2. Duplicates

In [ ]:
df.duplicated().sum()    # how many fully-duplicated rows are there?

In [ ]:
df = df.drop_duplicates()   # keeps the FIRST occurrence of each duplicate by default
df.duplicated().sum()

## 3. Data types

IDs are stored as integers, but they are **labels, not quantities** — you'd never take their mean. Store them as text so they aren't treated as numbers.

In [ ]:
id_cols = ["encounter_id", "patient_nbr", "admission_type_id",
           "discharge_disposition_id", "admission_source_id"]
df[id_cols] = df[id_cols].astype("string")
df[id_cols].dtypes

## 4. Renaming columns

Shorten a few unwieldy names (assign the result back — the modern idiom).

In [ ]:
df = df.rename(columns={
    "admission_type_id": "admit_type",
    "discharge_disposition_id": "discharge_dispo",
    "admission_source_id": "admit_source",
    "num_lab_procedures": "lab_procedures",
    "num_procedures": "procedures",
})
df.columns.tolist()[:10]

## 5. Missing data

Start by quantifying it: what fraction of each column is missing?

In [ ]:
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct.head(8)

Watch for **sentinel values** — this dataset uses `?` to mean missing. Those aren't counted as missing until you convert them to real `NaN`.

In [ ]:
(df == "?").sum().sort_values(ascending=False).head()

In [ ]:
df = df.replace("?", pd.NA)
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct.head(8)

A quick chart makes the picture obvious — which columns are mostly empty versus mostly complete.

In [ ]:
ax = missing_pct[missing_pct > 0].plot.barh(figsize=(7, 4))
ax.set_xlabel("% missing")
ax.set_title("Missing data by column")
plt.tight_layout(); plt.show()

## 6. Imputing

For a numeric column with a little missingness, filling with the **median** is a robust default (less sensitive to skew than the mean).

In [ ]:
df["num_medications"] = df["num_medications"].fillna(df["num_medications"].median())
df["num_medications"].isna().sum()

> Other options: `mean()`, a fixed constant, or — for ordered/time data — carry values forward/backward with `df.ffill()` / `df.bfill()`. (Note: the old `fillna(method='ffill')` and chained `fillna(..., inplace=True)` forms are deprecated/removed in modern pandas; assign the result back instead.)

## 7. Fixing categorical values

Inspect a category's values with `value_counts`, then standardize. Use vectorized string methods (`.str.lower()`), which skip `NaN` safely — unlike `apply(lambda x: x.lower())`, which errors on missing values.

In [ ]:
df["gender"].value_counts(dropna=False)

In [ ]:
df["gender"] = df["gender"].str.strip().str.lower()
df["gender"] = df["gender"].replace({"m": "male", "f": "female", "mle": "male",
                                     "unknown/invalid": pd.NA})
df["gender"].value_counts(dropna=False)

> A data-quality principle: map genuinely unknown values (`unknown/invalid`) to **missing**, not to a guessed category. Don't fabricate data while cleaning it.

In [ ]:
sns.countplot(x="gender", data=df)
plt.title("Gender after cleaning"); plt.show()

### The `.str` accessor

Vectorized string methods operate on a whole text column at once — the core toolkit for messy categorical and text data. A few of the most useful:

In [ ]:
# df["race"].str.upper().head(3).tolist()              # transform every value
# df["race"].str.contains("Afr", na=False).sum()       # rows whose race contains 'Afr'
# df["diag_1"].str.startswith("250", na=False).sum()   # primary diabetes dx (ICD-9 250.x)

> `.str` also covers `.split()`, `.replace()`, `.strip()`, and `.extract()` (with a regex) — everything you need to pull apart and standardize text. [Working with text data](https://pandas.pydata.org/docs/user_guide/text.html).

## 8. Distributions & outliers

A histogram shows a numeric column's shape; a boxplot makes outliers visible.

In [ ]:
sns.histplot(data=df, x="time_in_hospital", binwidth=1)
plt.title("Length of stay (days)"); plt.show()

In [ ]:
sns.boxplot(data=df, y="num_medications")
plt.title("Medications per encounter"); plt.show()

A common, defensible rule for flagging outliers is the **1.5 × IQR** rule (the same rule a boxplot's whiskers use).

In [ ]:
q1, q3 = df["num_medications"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
print(f"keep rows with num_medications in [{low:.0f}, {high:.0f}]")

before = len(df)
df = df[df["num_medications"].between(low, high)]
print("removed", before - len(df), "outlier rows")

### Correlations

A quick numeric scan of how the numeric columns move together — no plot required. `numeric_only=True` is needed because the frame still has text columns.

In [ ]:
df.corr(numeric_only=True).round(2)

> Values near +1 or −1 mean a strong linear relationship; near 0, little. (Rendering this as a heatmap is covered in the Data Visualization course.)

## 9. Dropping columns & rows

In [ ]:
# Drop columns that are mostly empty or not useful for analysis
df = df.drop(columns=["weight", "payer_code", "max_glu_serum"])
df.shape

In [ ]:
# Drop rows with an invalid 'age' sentinel we saw in the first look
df = df[df["age"] != "xyz"]
df.shape

### Exercise — Finish the cleaning *(12 min)*

Working on the current `df`:
1. After the `?` → `NaN` fix, how many rows are missing `race`?
2. Compute the percentage of `A1Cresult` that is missing. If it exceeds 35%, drop the column.
3. Report the final shape.

In [ ]:
# Your work here


## Wrap-up

You can inspect a dataset, remove duplicates, fix types, quantify and handle missing data (including sentinels), standardize categories, and deal with outliers — the core of getting data analysis-ready.

**Next:** Feature engineering & GroupBy — building new columns and summarizing by group.